In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from smt.sampling_methods import LHS


import os
import sys

# Changing directory to FEWNEW/work
# to import stuffs
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglike 
# import modeselectoralt
import parismc
# import gc
import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 3/12     # Total time
print(f"Using dt = {dt} seconds, T = {T} years")

print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_comb = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
}

sum_kwargs_sep = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
    "separate_modes": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu
)

# Kerr eccentric flux
waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)


print('Done initializing waveform generator.')

print("Creating GravWaveAnalysis class...")
gwf = GWfuncs.GravWaveAnalysis(T, dt)

print("Initializing loglike class...")


# Source parameters
m1 = 1e6
m2 = 1e1
a = 0.7
p0 = 9
e0 = 0.4
xI0 = 1.0
dist = 1.8  # Gpc
qS = np.pi
phiS = 0.
qK =  0.
phiK = 0.
Phi_phi0 = 0.4
Phi_theta0 = 0.0
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

# n-indexed mode selection parameters
n_vals = np.arange(-1,6)  # n from -1 to 5
ell = 2  # quadrupole only

# NOTE: change verbose argument for debugging
# Using n-indexed mode selection
loglike_obj = loglike.LogLike(
    params_star,
    waveform_gen_comb,
    gwf,
    verbose=False,
    waveform_gen_sep=waveform_gen_sep,
    ell=ell,
    n_vals=n_vals,
    M_mode=None  # No SNR filtering, use all n-groups
)

print('Done initializing loglike class.')
print('Calculating SNR...')
data = loglike_obj.signal
data_snr = gwf.rhostat(data)
print('SNR calculated:', data_snr)
print("Setting up log_density and prior functions...")

# REVERSE TEMP for annealing
# supposed to be 1/10 if we're going with the right nomenclature 
# so supposed to be reversetemp
temp = 10
def log_density(params):
    params = np.asarray(params)

    n_samples = params.shape[0] 
    log_likes = np.zeros(n_samples)


    for i in range(n_samples):
        logm1, logm2, a, p0, e0 = params[i]
        m1 = 10**logm1
        m2 = 10**logm2

        try:
            # NOTE: scaled by temp
            loglike = temp*loglike_obj(np.array([m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]))
        except Exception:
            loglike = -np.inf
        log_likes[i] = loglike

    return log_likes

def prior_transform(u):
    logm1lim = [5.6, 6.4]
    logm2lim = [0.8,1.3]
    alim = [0.3, 0.99]
    p0lim = [8.0, 11.0]
    e0lim = [0.2, 0.5]

    transformed = np.zeros_like(u)

    # Uniform in log for masses

    # m1
    transformed[:, 0] = (logm1lim[1] - logm1lim[0]) * u[:, 0] + logm1lim[0]

    # m2
    transformed[:, 1] = (logm2lim[1] - logm2lim[0]) * u[:, 1] + logm2lim[0]

    # Linear in others 

    # a
    transformed[:, 2] = (alim[1] - alim[0]) * u[:, 2] + alim[0]

    # p0
    transformed[:, 3] = (p0lim[1] - p0lim[0]) * u[:, 3] + p0lim[0] 

    # e0
    transformed[:, 4] = (e0lim[1] - e0lim[0]) * u[:, 4] + e0lim[0]

    
    return transformed

def inverse_prior_transform(params):
    logm1lim = [5.6, 6.4]
    logm2lim = [0.8, 1.3]
    alim = [0.3, 0.99]
    p0lim = [8.0, 11.0]
    e0lim = [0.2, 0.5]

    params = np.asarray(params)
    u = np.zeros_like(params)

    u[:, 0] = (params[:, 0] - logm1lim[0]) / (logm1lim[1] - logm1lim[0])
    u[:, 1] = (params[:, 1] - logm2lim[0]) / (logm2lim[1] - logm2lim[0])
    u[:, 2] = (params[:, 2] - alim[0]) / (alim[1] - alim[0])
    u[:, 3] = (params[:, 3] - p0lim[0]) / (p0lim[1] - p0lim[0])
    u[:, 4] = (params[:, 4] - e0lim[0]) / (e0lim[1] - e0lim[0])

    return u

print('Done setting up log-likelihood and prior.')
print('Setting up ParisMC sampler...')
config = parismc.SamplerConfig(
    merge_confidence=0.9,          # Coverage prob → Mahalanobis merge radius R_m (higher is more permissive)
    alpha=int(1e5),                    # Use recent samples for weighting. 
    trail_size=int(1e3),          # Maximum trials per iteration
    boundary_limiting=True,        # Enable boundary constraints
    use_beta=True,                # Use beta correction for boundaries
    integral_num=int(1e5),        # MC samples for beta estimation
    gamma=500,                    # Covariance update frequency NOTE: changed from 100
    exclude_scale_z=np.inf,       # No exclusion based on weights
    use_pool=False,               # Set to True for multiprocessing
    keep_dead_processes=True
)

print('Done setting up ParisMC sampler.')
print('Setting up initial covariance matrix...')

# Change to the search directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/search')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/search')


ndim = 5
n_seed = 1

inv_cov = np.array([[ 1.62413275e+006, -1.28443452e+006,  5.67130720e+005,
                    9.57067547e+005, -9.67768736e+005],
                    [-1.28443452e+006,  1.72045648e+006, -7.15930848e+005,
                    -1.02209029e+006,  1.46765877e+006],
                    [ 5.67130720e+005, -7.15930848e+005,  4.37737159e+005,
                    5.42371762e+005, -6.08570395e+005],
                    [ 9.57067547e+005, -1.02209029e+006,  5.42371762e+005,
                    7.47638444e+005, -8.38325090e+005],
                    [-9.67768736e+005,  1.46765877e+006, -6.08570395e+005,
                    -8.38325090e+005,  1.27861238e+006]])
init_cov_list = [np.linalg.inv(inv_cov)/temp for _ in range(n_seed)]

print('Done setting up initial covariance matrix.')

print('Initializing sampler...')
sampler = parismc.Sampler(
    ndim=ndim, 
    n_seed=n_seed,
    log_density_func=log_density,
    init_cov_list=init_cov_list,
    prior_transform=prior_transform,
    config=config
)
print('Done initializing sampler.')

Using dt = 10 seconds, T = 0.25 years
Initializing waveform generator...
Creating GenerateEMRIWaveform class...
Done initializing waveform generator.
Creating GravWaveAnalysis class...
Initializing loglike class...
Done initializing loglike class.
Calculating SNR...
SNR calculated: 5.942134448355127
Setting up log_density and prior functions...
Done setting up log-likelihood and prior.
Setting up ParisMC sampler...
Done setting up ParisMC sampler.
Setting up initial covariance matrix...
Done setting up initial covariance matrix.
Initializing sampler...
Done initializing sampler.


In [2]:

print('Using external LHS samples at previous best fit point...')
best_fit = [5.99924515, 1.00389791, 0.69519314, 9.02648957, 0.39784083]

external_lhs_points = inverse_prior_transform(np.array([best_fit]))
external_lhs_log_densities = log_density(prior_transform(external_lhs_points))
print('External LHS points:', external_lhs_points)
print('External LHS log densities:', external_lhs_log_densities)



Using external LHS samples at previous best fit point...
External LHS points: [[0.49905644 0.40779582 0.57274368 0.34216319 0.65946943]]
External LHS log densities: [1.38356639]


In [6]:
log_density([param_true])

array([59.07162615])

In [4]:
print('Running sampling...')
def save_every_1000(sampler, i):
    if i % 1000 == 0 and i > 0:
        sampler.save_state()

sampler.run_sampling(
    num_iterations=int(1e4),
    savepath='./intrinsic_ffunc_3mth_snr32_run4_nt',
    print_iter=100, # Print progress every n iterations
    callback=save_every_1000,
    external_lhs_points=external_lhs_points,
    external_lhs_log_densities=external_lhs_log_densities
)
print('Done running sampling.')

Running sampling...


Sampling:   0%|          | 0/10000 [00:00<?, ?it/s]

Detected problematic log_density (NaN or Inf) 1000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 2000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 3000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 4000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 5000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 6000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 7000 times. Check your implementation.
Detected problematic log_density (NaN or Inf) 8000 times. Check your implementation.


Done running sampling.


In [8]:
proc_pt = sampler.searched_points_list
proc_pt

[array([[ 0.49905644,  0.40779582,  0.57274368,  0.34216319,  0.65946943],
        [ 0.49331089,  0.40529959,  0.55461364,  0.36293934,  0.66297377],
        [ 0.48677474,  0.39323397,  0.54642468,  0.37763402,  0.6780503 ],
        ...,
        [-0.02066735,  1.43080746,  1.49286208,  0.97442709,  0.9106043 ],
        [ 1.32739755, -0.87236301,  0.8244815 ,  0.2556193 ,  0.32150873],
        [-0.04849275,  1.38461227,  0.3569086 ,  0.55304147,  0.80644972]],
       shape=(10999, 5))]

In [7]:
logden_list = sampler.searched_log_densities_list
logden_list

[array([1.38356639, 0.16022114, 0.14221533, ...,       -inf,       -inf,
              -inf], shape=(10999,))]

In [9]:
maxld_pt1 = prior_transform(proc_pt[0][np.argmax(logden_list)].reshape(1, -1))


In [10]:
maxld_pt1

array([[5.96243941, 0.98135902, 0.61653963, 9.45218584, 0.41423134]])